# TP 4 — Partie 3 : écrire le flux dans la couche Bronze

**Objectif :** que chaque message du topic `suivi_colis` arrive dans la table Iceberg `bronze.suivi_evenements`, **une seule fois**, même si la requête s'arrête en plein flux.

Comme au TP 1, la couche Bronze garde les données **telles qu'elles sont arrivées** : la valeur du message reste du texte JSON, non interprété. On y ajoute ce que Kafka sait du message : sa partition, son offset et son heure d'arrivée.

| Colonne | Contenu |
|---|---|
| `cle` | la clé du message (`colis_id`) |
| `valeur` | le message, en texte JSON |
| `kafka_partition`, `kafka_offset` | sa position dans le topic : identifiant unique du message |
| `recu_le` | l'heure à laquelle Kafka l'a reçu |

> **⚠️ Un seul notebook actif à la fois**, et le profil `streaming` démarré (partie 2).

In [ ]:
import time

from pyspark.sql import SparkSession, functions as F

from flux import (CHECKPOINTS, KAFKA, TOPIC, a_completer, arreter_flux, creer_topic, fin_du_rejeu,
                  lister_s3, offsets_du_checkpoint, rejouer, suivre, vider_s3)

spark = (SparkSession.builder.appName("tp4-bronze")
         .config("spark.ui.showConsoleProgress", "false")   # pas de barres de progression
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")     # n'affiche que les erreurs de Spark
CHECKPOINT = f"{CHECKPOINTS}/bronze"

# On repart de zéro : topic vide, table et checkpoint supprimés
arreter_flux(spark)
creer_topic(TOPIC)
spark.sql("CREATE NAMESPACE IF NOT EXISTS lakehouse.bronze")
spark.sql("DROP TABLE IF EXISTS lakehouse.bronze.suivi_evenements PURGE")
vider_s3(CHECKPOINT)
spark.sql("""CREATE TABLE lakehouse.bronze.suivi_evenements (
    cle STRING, valeur STRING, kafka_partition INT, kafka_offset BIGINT, recu_le TIMESTAMP) USING iceberg""")
print("prêt")

## Étape 1 — Lancer le flux

Le rejeu envoie tout le mois de mars 2018, à raison d'un jour par seconde : il dure environ 30 secondes et tourne en arrière-plan pendant les étapes suivantes. Si vous êtes trop lent, ce n'est pas grave : les messages attendent dans Kafka.

In [ ]:
rejeu = rejouer("--depuis", "2018-03-01", "--jusqu-a", "2018-04-01", "--jours-par-seconde", 1,
                attendre=False)

## Étape 2 — Lire Kafka avec Spark

Spark lit Kafka avec `spark.readStream.format("kafka")` et trois options :
- `kafka.bootstrap.servers` : l'adresse du broker (`KAFKA`) ;
- `subscribe` : le topic (`TOPIC`) ;
- `startingOffsets` : `earliest`, pour lire depuis le début du topic au premier démarrage.

Le DataFrame obtenu a toujours les mêmes colonnes : `key` et `value` (en octets), `topic`, `partition`, `offset`, `timestamp`…

Créez `flux`, qui lit le topic puis ne garde que les colonnes de la table Bronze : `key` et `value` converties en texte (`.cast("string")`), et renommées comme dans le tableau ci-dessus.

In [ ]:
flux = a_completer("flux")  # À COMPLÉTER
flux.printSchema()

## Étape 3 — Écrire en continu dans Iceberg

Complétez la fonction `demarrer_bronze()`. Elle démarre l'écriture de `flux` dans la table et renvoie la requête :
- `writeStream`, en mode `append` (on ne fait qu'ajouter des lignes) ;
- un micro-lot toutes les 5 secondes : `.trigger(processingTime="5 seconds")` ;
- le **checkpoint** : `.option("checkpointLocation", CHECKPOINT)` ;
- `.toTable("lakehouse.bronze.suivi_evenements")`, qui démarre la requête.

Le **checkpoint** est un dossier où Spark note, à chaque micro-lot, les offsets qu'il a lus et ceux qu'il a écrits. C'est sa mémoire.

In [ ]:
def demarrer_bronze():
    return a_completer("demarrer_bronze")  # À COMPLÉTER


requete = demarrer_bronze()
suivre(requete, secondes=15)

## Étape 4 — Une panne en plein flux

On arrête la requête, comme si le serveur Spark tombait en panne. Le rejeu, lui, continue : les transporteurs n'attendent pas.

In [ ]:
requete.stop()
avant = spark.table("lakehouse.bronze.suivi_evenements").count()
print("lignes en Bronze au moment de la panne :", avant)
time.sleep(10)
print("10 s plus tard, toujours :", spark.table("lakehouse.bronze.suivi_evenements").count())

On redémarre **exactement le même code**, avec le même checkpoint.

In [ ]:
requete = demarrer_bronze()
suivre(requete, secondes=10)

On attend la fin du rejeu et la lecture des derniers messages, puis on vérifie la table.

In [ ]:
n_envoyes = fin_du_rejeu(rejeu)
requete.processAllAvailable()
requete.stop()

bronze = spark.table("lakehouse.bronze.suivi_evenements")
n_bronze = bronze.count()
n_positions = bronze.select("kafka_partition", "kafka_offset").distinct().count()
print(f"envoyés : {n_envoyes} | en Bronze : {n_bronze} | positions Kafka distinctes : {n_positions}")

> **❓ Question 1.** Le premier micro-lot après le redémarrage est plus gros que les autres : pourquoi ? Combien de messages ont été perdus, et combien écrits deux fois ? Comment Spark a-t-il su où reprendre ?

## Étape 5 — Dans le checkpoint

Le checkpoint est un simple dossier dans RustFS. Regardons son contenu, puis le dernier fichier d'offsets.

In [ ]:
for cle, taille in lister_s3(CHECKPOINT.replace("s3a://", "s3://"))[:12]:
    print(f"{cle:60} {taille:>6} octets")
lot, offsets = offsets_du_checkpoint(CHECKPOINT)
print(f"\nmicro-lot {lot} : jusqu'aux offsets {offsets}")

> **❓ Question 2.** Comparez les offsets du checkpoint au nombre de messages de chaque partition (Kafka UI, onglet *Overview*). Que se passerait-il si l'on supprimait le checkpoint avant de relancer la requête ? Et si on le gardait, mais qu'on écrivait dans une autre table ?

## Étape 6 — Le prix du flux : les petits fichiers

Chaque micro-lot non vide produit un **commit** Iceberg, donc un snapshot et de nouveaux fichiers.

In [ ]:
spark.sql("""SELECT count(*) AS snapshots FROM lakehouse.bronze.suivi_evenements.snapshots""").show()
spark.sql("""SELECT count(*) AS fichiers, round(avg(file_size_in_bytes) / 1024) AS taille_moyenne_ko
             FROM lakehouse.bronze.suivi_evenements.files""").show()

> **❓ Question 3.** Combien de fichiers pour un mois de suivi ? Combien en aurait-on au bout d'un an, avec un micro-lot toutes les 5 secondes ? Quel outil du TP 3 règle ce problème ?

## À retenir

- Spark lit Kafka comme une table qui grandit : `key`, `value`, `partition`, `offset`, `timestamp`.
- Le **checkpoint** mémorise les offsets traités : après une panne, la requête reprend là où elle s'était arrêtée, sans perte ni doublon.
- Chaque micro-lot crée des fichiers : un flux doit être compacté.